![DB Academy](../Includes/images/common/db-academy.png)

# 02L - Deploy a Simple Declarative Automation Bundle (DAB)

### Estimated Duration: ~15 minutes

## Overview

In this lab, you'll take a notebook project shared by a coworker and deploy it to a development environment using a **Declarative Automation Bundle (DAB)**. You'll build the source job in the UI, capture its YAML, finish the bundle's **databricks.yml** configuration, then validate, deploy, run, verify, and destroy the bundle using the Databricks CLI.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Generate a YAML job configuration** by building a job in the UI and using **View as code**.
2. **Update a `databricks.yml` file** with your job and a `dev` target that uses `mode: development`.
3. **Validate, deploy, run, and destroy** a bundle with `databricks bundle` CLI commands.
4. **Verify** that your deployed job actually produced the expected data.

## Reference Documentation

- **What are bundles?** (intro): [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/)
- **Bundle configuration reference (full YAML key list)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/reference) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/reference) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/reference)
- **Bundle settings (mappings)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/settings) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/settings) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/settings)
- **`databricks bundle` CLI commands**: [AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)
- **Variable substitution (`${workspace.…}`, `${bundle.…}`)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)
- **Deployment modes**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes)
- **Create and manage jobs**: [AWS](https://docs.databricks.com/aws/en/jobs/create-run-jobs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/jobs/create-run-jobs) | [GCP](https://docs.databricks.com/gcp/en/jobs/create-run-jobs)

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>



## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was setup using the **0 - REQUIRED - Course Setup and Authentication**.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **0 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and data for your environment.

  </div>
</div>



## A. Classroom Setup

Run the following cell to configure your working environment for this course.

In [0]:
%run ../Includes/Classroom-Setup-02L

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15884059_1784198598'.


DATABRICKS_HOST set to:  https://dbc-7f5587c4-8fc8.cloud.databricks.com
DATABRICKS_TOKEN set.


Installed Databricks CLI v0.298.0 at /root/bin/databricks.


Catalog check for the labs passed.
Created the nyctaxi_dev table in your dev catalog: labuser15884059_1784198598_1_dev!


Information,Value
DEV catalog reference: DA.catalog_dev:,labuser15884059_1784198598_1_dev


Compute,Status,Details
All-Purpose,✓ Match,Version 17.3


## B. Lab Scenario

You are responsible for deploying Databricks projects through your organization's CI/CD process using **Declarative Automation Bundles (DABs)**.

A coworker shared a notebook located in `./src/our_project_code`.

Your task is to begin the deployment process by configuring and deploying the project to the **development** environment.

**The provided notebook:**

- Reads from the development dataset **nyctaxi_raw**
- Uses your **labuser_UNIQUE_ID_1_dev.default** catalog
- Creates a simple bronze and silver table pipeline
- Uses job parameters to define the development catalog

**To complete this lab, you will:**

- Retrieve the job's YAML configuration
- Update the **databricks.yml** bundle configuration file
- Deploy the bundle from the **02L - Deploy a Simple DAB** folder

## C. Preview the Development Data
1. Preview the **nyctaxi_raw** data in your **labuser_UNIQUE_ID_1_dev** catalog.

    Notice that the development data contains a small sample of the production data (100 rows).

In [0]:
spark.sql(f'''
SELECT *
FROM {catalog_dev}.default.nyctaxi_raw
''').display()

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53Z,2016-02-13T21:57:15Z,1.4,8.0,10103,10110
2016-02-13T18:29:09Z,2016-02-13T18:37:23Z,1.31,7.5,10023,10023
2016-02-06T19:40:58Z,2016-02-06T19:52:32Z,1.8,9.5,10001,10018
2016-02-12T19:06:43Z,2016-02-12T19:20:54Z,2.3,11.5,10044,10111
2016-02-23T10:27:56Z,2016-02-23T10:58:33Z,2.6,18.5,10199,10022
2016-02-13T00:41:43Z,2016-02-13T00:46:52Z,1.4,6.5,10023,10069
2016-02-18T23:49:53Z,2016-02-19T00:12:53Z,10.4,31.0,11371,10003
2016-02-18T20:21:45Z,2016-02-18T20:38:23Z,10.15,28.5,11371,11201
2016-02-03T10:47:50Z,2016-02-03T11:07:06Z,3.27,15.0,10014,10023
2016-02-19T01:26:39Z,2016-02-19T01:40:01Z,4.42,15.0,10003,11222


## D. Pre-flight Checks

Before starting the lab tasks, run a few quick checks to confirm the Databricks CLI is installed, authenticated, and pointed at the right working directory.

### D1. Check the Databricks CLI Version

Run a CLI command to confirm the Databricks CLI version is **v0.298.0**.

In [0]:
%sh
databricks -v

Databricks CLI v0.298.0


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks -v
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### D2. Confirm CLI Authentication

Run the cell below to confirm the Databricks CLI is authenticated against your workspace. If authentication is broken, the cell will return an error rather than a list of catalogs.

In [0]:
%sh
databricks catalogs list

Name                    Type                  Comment
dbacademy               MANAGED_CATALOG       
dbacademy_cdc_diabetes  DELTASHARING_CATALOG  # CDC Diabetes Health Indicators

## Attribution

This course uses the CDC Diabetes Health Indicators Dataset, which is licensed under CC0: Public Domain.

## License Information

You can find more details about the "CDC Diabetes Health Indicators Dataset" on [Kaggle](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset) and[UC Irvine ML Repository](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)

The "CDC Diabetes Health Indicators Dataset" is distributed under the CC0: Public Domain license. Please review the license terms before using the dataset.

For any questions or inquiries regarding the dataset or its usage, please refer to the above pages for contact information.

---
Note: This README is provided as part of the dataset attribution for educational purposes. Please ensure compliance w

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    DATABRICKS CLI ERROR TROUBLESHOOTING:
  </strong>
  <div style="color:#333;">

  - If you encounter a Databricks CLI authentication error, it means the authentication was not successful. Confirm you ran the notebook using your **all purpose compute**.

  - If you encounter the error below, it means your **databricks.yml** file has syntax issues due to a modification. Even for non-DAB CLI commands, the **databricks.yml** file is still required, as it may contain important authentication details, such as the host and profile, which are utilized by the CLI commands.

![CLI Invalid YAML](../Includes/images/databricks_cli_error_invalid_yaml.png)
  </div>
</div>


### D3. List the Working Directory

Use the `ls` command to view the available files in the current directory. 

  Confirm that you see the **databricks.yml** file, the **src** folder, and this lab notebook.

In [0]:
%sh
ls

Lab - Deploy a Simple DAB.ipynb
databricks.yml
solution
src


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
ls
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>


## E. Task 1 - Get the Job's YAML Configuration

In this task, you'll build the source job in the UI, then export its YAML using **View as code**. 

This is the same flow used in the previous demo.

### Step 1.1 - Get Your Cluster ID

Run the following cell to obtain your cluster ID for the lab. You will see this in the YAML configuration file.

**NOTE:** If you select your cluster when creating the job in the UI, the cluster ID will already be present in the generated YAML. The cell below is a fallback so you have it on hand.

In [0]:
spark.conf.get("spark.databricks.clusterUsageTags.clusterId")

'0716-104408-ubose5n6'

### Step 1.2 - Build the Job in the UI

Manually create the job that the bundle will deploy. The fastest way to get a correct YAML configuration is to build the job in the UI and then export it.

**Job requirements:**

a. Name the job `lab02_job_yourfirstname` (replace with your first name).

b. Add a single notebook task with the following:

| Configuration | Value |
|---|---|
| **Task Name** | `create_nyc_tables` |
| **Notebook** | `./02L - Deploy a Simple DAB/src/our_project_code` |
| **Compute** | Use your current lab cluster for the job's compute. Selecting the cluster automatically includes the cluster ID in the generated YAML. Using all-purpose compute for jobs is **not a best practice**. This is for training purposes only. |

Then select **Create task**.

c. Add the following **Job parameters**. Make sure these are added at the **job level**, not the task level.

| Parameter | Value |
|---|---|
| `catalog_name` | Reference the catalog shown in **A. Classroom Setup** (your `labuser_UNIQUE_ID_1_dev` catalog) |
| `display_target` | `Development` |

**Reference:** Create and manage jobs documentation:
[AWS](https://docs.databricks.com/aws/en/jobs/create-run-jobs) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/jobs/create-run-jobs) |
[GCP](https://docs.databricks.com/gcp/en/jobs/create-run-jobs)

### Step 1.3 - Copy the YAML via 'View as code'

Once the job is created, click the kebab menu (three vertical dots) near **Run now** and select **View as code**. Choose the **YAML** format and click **Copy**.

**NOTE:** You can also test the configuration by running the job from the UI before continuing.

**Reference:** View jobs as code:
[AWS](https://docs.databricks.com/aws/en/jobs/automate#view-jobs-as-code) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/jobs/automate#view-jobs-as-code) |
[GCP](https://docs.databricks.com/gcp/en/jobs/automate#view-jobs-as-code)

## F. Task 2 - Update the **databricks.yml** File

Modify the **databricks.yml** configuration file in the **02L - Deploy a Simple DAB** folder.

### Step 2.1 - Paste your job under `resources.jobs`

Add the YAML configuration you copied in Step 1.3 under the `RESOURCES` comment in the **databricks.yml** file.

### Step 2.2 - Convert to a relative notebook path

In the pasted job configuration, modify the `notebook_path` so it uses a relative path and includes the correct file extension.

- The source file is `our_project_code.ADD_CORRECT_EXTENSION`, so the YAML must reference the file `./src/our_project_code`.
  - Make sure to specify the correct extension.

### Step 2.3 - Add the `dev` target

In the `targets` mapping, add a target named `dev` with the following:

- `default: true` (so `dev` is the default target if `-t` is omitted)
- `mode: development` (development mode prepends a `[dev <user>]` prefix, pauses schedules, and tags resources with a `dev` tag)
- A `workspace.root_path` that uses variable substitution:

```yaml
workspace:
  root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
```

**References:**
- Bundle settings (mappings):
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/settings) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/settings) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/settings)
- Variable substitution:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)
- Deployment modes:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes)


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    HINT
  </strong>
  <div style="color:#333;">

If you get stuck, an example **databricks.yml** solution is in the accompanying **solution** folder.
  </div>
</div>



## G. Task 3 - Validate the Bundle

Validate your **databricks.yml** bundle configuration file using the Databricks CLI. 

  Run the cell and confirm validation succeeds. If there is an error, fix the **databricks.yml** file and re-run.

**HINT:** See the `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
%sh 
databricks bundle validate

Name: demo02_lab_bundle
Target: dev
Workspace:
  User: labuser15884059_1784198598@vocareum.com
  Path: /Workspace/Users/labuser15884059_1784198598@vocareum.com/.bundle/demo02_lab_bundle/dev

Validation OK!


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle validate
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>



<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    TROUBLESHOOTING Common Validation Issues:
  </strong>
  <div style="color:#333;">

  - Path or extention of your notebook is incorrect.
  - Incorrect mappings under `target`. 

  </div>
</div>


## H. Task 4 - Deploy the Bundle to the `dev` Target

Deploy the bundle to the development target using the Databricks CLI. After the cell completes, navigate to **Jobs & Pipelines** and confirm a job named **[dev <user>] lab02_job_<firstname>** was created.

Specifically, verify:

- The task references the correct notebook (`./src/our_project_code.ipynb`).
- The job parameters reference your `labuser_UNIQUE_ID_1_dev` catalog.

**NOTE:** Deployment will take about a minute to complete.

**HINT:** See the `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
%sh
databricks bundle deploy -t dev

Source-linked deployment is enabled. Deployed resources reference the source files in your working tree instead of separate copies.
Deploying resources...
Updating deployment state...
Deployment complete!


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle deploy -t dev
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>




## I. Task 5 - Run the Bundle

Run the deployed job using the Databricks CLI.

**NOTE:** This will take 1-2 minutes to complete.

**HINT:** Use the **job key** from the `resources` mapping in your **databricks.yml** file (your name will differ):

```
resources:
  jobs:
    lab02_job_yourfirstname:    # <--- This is the job key
      name: lab02_job_yourfirstname
```

**Reference:** `databricks bundle` CLI commands:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
%sh
databricks bundle run lab02_job_tymur_hilf

Run URL: https://dbc-7f5587c4-8fc8.cloud.databricks.com/?o=7474652962058183#job/432367104452920/run/341195977384838

2026-07-16 12:34:43 "[dev labuser15884059_1784198598] lab02_job_tymur_hilf" RUNNING
2026-07-16 12:35:42 "[dev labuser15884059_1784198598] lab02_job_tymur_hilf" TERMINATED SUCCESS


Output:
Task create_bronze_table:

Task create_silver_table:



##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
## Instead of labuser123 use your own job key
databricks bundle run -t dev lab02_job_labuser123
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>




## J. Task 6 - Verify the Bronze Table

After the job completes, run the cell below to confirm the bronze table was created correctly.

**The job created two tables in your `labuser_UNIQUE_ID_1_dev` catalog:**

- **nyctaxi_bronze**
- **nyctaxi_silver**

The cell below checks the row count of the bronze table.

In [0]:
check_nyctaxi_bronze_table(user_catalog = catalog_dev, total_count=100)

The nyctaxi_bronze table has was created successfully from your DAB deployment!


## K. Task 7 - Destroy the Bundle

You're done with this lab, so destroy the bundle to clean up the deployed job and artifacts.

**HINT:** See the `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
%sh
databricks bundle destroy --auto-approve

The following resources will be deleted:
  delete resources.jobs.lab02_job_tymur_hilf

All files and directories at the following location will be deleted: /Workspace/Users/labuser15884059_1784198598@vocareum.com/.bundle/demo02_lab_bundle/dev

Deleting files...
Destroy complete!


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
databricks bundle destroy --auto-approve
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>





## Conclusion

Nice work. In this lab you took a notebook project shared by a coworker and went all the way through the bundle lifecycle:

1. Built the source job in the UI and exported its YAML using **View as code**.
2. Updated the **databricks.yml** with your job under `resources.jobs` and added a `dev` target with `mode: development` and a templated `workspace.root_path`.
3. Validated, deployed, and ran the bundle with `databricks bundle validate`, `databricks bundle deploy -t dev`, and `databricks bundle run -t dev <job_key>`.
4. Verified the deployed job actually created the **nyctaxi_bronze** table with the expected row count.
5. Cleaned up by destroying the bundle with `databricks bundle destroy --auto-approve`.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>